# Traffic Demand Prediction
This notebook demonstrates a complete end-to-end Machine Learning pipeline for predicting spatiotemporal traffic demand using **CatBoost**.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

# Load Data
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')
print(f"Train shape: {train.shape}, Test shape: {test.shape}")
train.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("Missing Values:")
print(train.isnull().sum())

plt.figure(figsize=(8,5))
sns.histplot(train['demand'].dropna().sample(min(10000, len(train))), bins=50)
plt.title('Sampled Demand Distribution')
plt.show()

## 3. Spatiotemporal Feature Engineering
Traffic demand is highly cyclical. We extract time from timestamps and convert them into continuous cyclic features (Sine and Cosine transforms) to capture the 24-hour cycle.

In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # Temporal Features
    if 'timestamp' in df.columns:
        time_split = df['timestamp'].str.split(':', expand=True)
        df['hour'] = time_split[0].astype(int)
        df['minute'] = time_split[1].astype(int)
        
        # Cyclical mapping
        df['time_in_mins'] = df['hour'] * 60 + df['minute']
        df['sin_time'] = np.sin(2 * np.pi * df['time_in_mins'] / 1440)
        df['cos_time'] = np.cos(2 * np.pi * df['time_in_mins'] / 1440)
        df.drop(['timestamp', 'time_in_mins'], axis=1, inplace=True)
        
    # Impute Missing Values
    for col in ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
        if col in df.columns: df[col] = df[col].fillna('Unknown')
    for col in ['NumberofLanes', 'Temperature']:
        if col in df.columns: df[col] = df[col].fillna(df[col].median())
        
    return df

train_proc = engineer_features(train)
test_proc = engineer_features(test)
train_proc.head()

## 4. Modeling with CatBoost
CatBoost handles high-cardinality categoricals like `geohash` naturally.

In [ ]:
target = 'demand'
features = [c for c in train_proc.columns if c not in ['Index', target]]
cat_features = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

model = CatBoostRegressor(
    iterations=1500, learning_rate=0.05, depth=8, l2_leaf_reg=3,
    loss_function='RMSE', cat_features=cat_features, verbose=200, random_seed=42
)

# In practice, use train_test_split. For final submission, train on all data.
model.fit(train_proc[features], train_proc[target])

## 5. Generating Predictions

In [ ]:
preds = model.predict(test_proc[features])
preds = np.clip(preds, 0, None)

submission = pd.DataFrame({'Index': test['Index'], 'demand': preds})
submission.to_csv('submission_catboost.csv', index=False)
print("Saved predictions to submission_catboost.csv!")